In [53]:
import jpy_tools.parseSnake2 as jps
import pandas as pd

In [54]:
snakeFile = jps.SnakeFile()
snakeHeader = jps.SnakeHeader(snakeFile, "/datapool/data/Users/zhijian/github/jpy_tools/pipeline/mulocdeep/config.yaml")

In [55]:
config = snakeHeader.getConfig()
snakeHeader

import pandas as pd
#configfile: "/datapool/data/Users/zhijian/github/jpy_tools/pipeline/mulocdeep/config.yaml"
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"

In [56]:
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"

In [57]:
df_splitFa = pd.DataFrame(index=['AllFa'])
df_splitFa = df_splitFa.assign(
    resultDir='ResultDir/',
    pepFa=config['pepFa'],
    threads=config['threads'],
    pipelineDir=pipelineDir,
)
df_splitFa

,resultDir,pepFa,threads,pipelineDir
AllFa,ResultDir/,/datapool/home/zhijian/data/wheat/cs_2_1/downl...,24,/datapool/data/Users/zhijian/github/jpy_tools/...


In [58]:
rule_splitFa = jps.SnakeRule(snakeFile, "splitFa", wildCard='sample', threads=1)
rule_splitFa.addCode("""
df_splitFa = pd.DataFrame(index=['AllFa'])
df_splitFa = df_splitFa.assign(
    resultDir='ResultDir/',
    pepFa=config['pepFa'],
    threads=config['threads'],
    pipelineDir=pipelineDir,
)
df_splitFa
                    """)
rule_splitFa.addMetaDf('df_splitFa', ['resultDir'], df_splitFa)
rule_splitFa.addMain('input', ['pepFa'])
rule_splitFa.addMain('params', ['threads', 'pipelineDir', 'resultDir'])
rule_splitFa.setShell("cd {pipelineDir} && bash split_fa.sh {input.pepFa} {params.threads} {params.resultDir}")
rule_splitFa

2026-03-27 09:58:38.985 | INFO     | jpy_tools.parseSnake2:addRule:55 - splitFa step num: 1



## get parameter of rule `splitFa` ##
df_splitFa = pd.DataFrame(index=['AllFa'])
df_splitFa = df_splitFa.assign(
    resultDir='ResultDir/',
    pepFa=config['pepFa'],
    threads=config['threads'],
    pipelineDir=pipelineDir,
)
df_splitFa
                    
for column in ['resultDir']:
    df_splitFa[column] = resultDir + 'step1_splitFa/' + df_splitFa[column]
----------------
IN RULE
----------------
# parameter's dataframe of splitFa: 
# |       | resultDir   | pepFa                                                                                                                                                                                                     |   threads | pipelineDir                                                               |
# |:------|:------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-------

In [59]:
df_mulocdeep = pd.DataFrame(index=range(1, config["threads"] + 1))
df_mulocdeep.index = df_mulocdeep.index.astype(str)
df_mulocdeep = df_mulocdeep.assign(
    splitId=lambda _: _.index,
    resultMuloc=lambda _: _.index.astype(str) + "/",
    species=config["species"],
    mulocdeepDir=config["mulocdeepDir"],
    sample='AllFa'
)
df_mulocdeep.head(5)

,splitId,resultMuloc,species,mulocdeepDir,sample
1,1,1/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
2,2,2/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
3,3,3/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
4,4,4/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
5,5,5/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa


In [60]:
rule_mulocDeep = jps.SnakeRule(snakeFile, "mulocDeep", threads=1, wildCard='splitId', conda='mulocdeep')
rule_mulocDeep.addCode("""
df_mulocdeep = pd.DataFrame(index=range(1, config["threads"] + 1))
df_mulocdeep.index = df_mulocdeep.index.astype(str)
df_mulocdeep = df_mulocdeep.assign(
    splitId=lambda _: _.index,
    resultMuloc=lambda _: _.index.astype(str) + "/",
    species=config["species"],
    mulocdeepDir=config["mulocdeepDir"],
    sample='AllFa'
)
df_mulocdeep.head(5)
                    """)
rule_mulocDeep.addMetaDf('df_mulocdeep', ['resultMuloc'], df_mulocdeep)
rule_mulocDeep.addMain('params', ['resultDir'], fromRule=rule_splitFa)
rule_mulocDeep.addMain('params', ['splitId', 'resultMuloc', 'mulocdeepDir', 'species'])
rule_mulocDeep.setShell("cd {params.mulocdeepDir} && python predict.py -input {params.resultDir}/{params.splitId}.fa -output {params.resultMuloc} -species {params.species}")
rule_mulocDeep

2026-03-27 09:58:39.092 | INFO     | jpy_tools.parseSnake2:addRule:55 - mulocDeep step num: 2



## get parameter of rule `mulocDeep` ##
df_mulocdeep = pd.DataFrame(index=range(1, config["threads"] + 1))
df_mulocdeep.index = df_mulocdeep.index.astype(str)
df_mulocdeep = df_mulocdeep.assign(
    splitId=lambda _: _.index,
    resultMuloc=lambda _: _.index.astype(str) + "/",
    species=config["species"],
    mulocdeepDir=config["mulocdeepDir"],
    sample='AllFa'
)
df_mulocdeep.head(5)
                    
for column in ['resultMuloc']:
    df_mulocdeep[column] = resultDir + 'step2_mulocDeep/' + df_mulocdeep[column]

def parseDfToInput_mulocDeep_splitFa(wildcard):
    selfWildCardUnique = True
    if isinstance(df_mulocdeep.at[wildcard.splitId, 'sample'], list):
        selfWildCardUnique = False
    if selfWildCardUnique:
        return resultDir + 'step1_splitFa/' + df_mulocdeep.at[wildcard.splitId, 'sample'] + '.finished'
    else:
        return [resultDir + 'step1_splitFa/' + x + '.finished' for x in df_mulocdeep.loc[wildcard.splitId, 'sample']]

def parseDfToParams_mulocDeep

In [61]:
df_mulocdeep

,splitId,resultMuloc,species,mulocdeepDir,sample
1,1,1/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
2,2,2/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
3,3,3/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
4,4,4/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
5,5,5/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
6,6,6/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
7,7,7/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
8,8,8/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
9,9,9/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa
10,10,10/,viridiplantae,/datapool/data/Users/zhijian/softwares/MulocDe...,AllFa


In [62]:
df_mergeRes = (
    df_mulocdeep.groupby("sample")["splitId"]
    .agg(list)
    .pipe(pd.DataFrame)
    .rename(columns={"index": "sampleSplit"})
    .assign(dir_out=lambda df: df.index + "/", mulocDir="/")
)
df_mergeRes

,splitId,dir_out,mulocDir
sample,,,
AllFa,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",AllFa/,/


In [63]:
rule_mergeRes = jps.SnakeRule(snakeFile, "mergeRes", threads=1)
rule_mergeRes.addCode(
    """
df_mergeRes = (
    df_mulocdeep.groupby("sample")["splitId"]
    .agg(list)
    .pipe(pd.DataFrame)
    .rename(columns={"index": "sampleSplit"})
    .assign(dir_out=lambda df: df.index + "/", mulocDir="/")
)
df_mergeRes
                    """
)
rule_mergeRes.addMetaDf("df_mergeRes", ["dir_out"], df_mergeRes)
rule_mergeRes.addMetaDf("df_mergeRes", ["mulocDir"], rule=rule_mulocDeep)
rule_mergeRes.addMain("params", ["splitId"], fromRule=rule_mulocDeep)
rule_mergeRes.addMain("params", ["dir_out", "mulocDir"])
rule_mergeRes.setShell(
    "cd {params.mulocDir} && cat ./*/sub_cellular_prediction.txt > {params.dir_out}/cellular.txt && cat ./*/sub_organellar_prediction.txt > {params.dir_out}/organellar.txt"
)
rule_mergeRes

2026-03-27 09:58:39.294 | INFO     | jpy_tools.parseSnake2:addRule:55 - mergeRes step num: 3


Rule is specified, only add rule dir to specified columns



## get parameter of rule `mergeRes` ##
df_mergeRes = (
    df_mulocdeep.groupby("sample")["splitId"]
    .agg(list)
    .pipe(pd.DataFrame)
    .rename(columns={"index": "sampleSplit"})
    .assign(dir_out=lambda df: df.index + "/", mulocDir="/")
)
df_mergeRes
                    
for column in ['dir_out']:
    df_mergeRes[column] = resultDir + 'step3_mergeRes/' + df_mergeRes[column]
for column in ['mulocDir']:
    df_mergeRes[column] = resultDir + 'step2_mulocDeep/' + df_mergeRes[column]

def parseDfToInput_mergeRes_mulocDeep(wildcard):
    selfWildCardUnique = True
    if isinstance(df_mergeRes.at[wildcard.sample, 'splitId'], list):
        selfWildCardUnique = False
    if selfWildCardUnique:
        return resultDir + 'step2_mulocDeep/' + df_mergeRes.at[wildcard.sample, 'splitId'] + '.finished'
    else:
        return [resultDir + 'step2_mulocDeep/' + x + '.finished' for x in df_mergeRes.loc[wildcard.sample, 'splitId']]

def parseDfToParams_mergeRes_mulocDeep_splitId(wildcard):
 

In [64]:
rule_all = jps.SnakeAll(snakeFile, rule_mergeRes)
rule_all

rule all:
    input:
        mergeResFinished = [resultDir + 'step3_mergeRes/' + "" + sample + ".finished" for sample in df_mergeRes.index],

In [65]:
snakeFile.getMain('/datapool/data/Users/zhijian/github/jpy_tools/pipeline/mulocdeep/snakefile')

import pandas as pd
#configfile: "/datapool/data/Users/zhijian/github/jpy_tools/pipeline/mulocdeep/config.yaml"
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"


## get parameter of rule `splitFa` ##
df_splitFa = pd.DataFrame(index=['AllFa'])
df_splitFa = df_splitFa.assign(
    resultDir='ResultDir/',
    pepFa=config['pepFa'],
    threads=config['threads'],
    pipelineDir=pipelineDir,
)
df_splitFa
                    
for column in ['resultDir']:
    df_splitFa[column] = resultDir + 'step1_splitFa/' + df_splitFa[column]


## get parameter of rule `mulocDeep` ##
df_mulocdeep = pd.DataFrame(index=range(1, config["threads"] + 1))
df_mulocdeep.index = df_mulocdeep.index.astype(str)
df_mulocdeep = df_mulocdeep.assign(
    splitId=lambda _: _.index,
    resultMuloc=lambda _: _.index.astype(str) + "/",
    species=config["species"],
    mulocdeepDir=config["mulocdeepDir"],
    sample='AllFa'
)
df_mul